In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

# Settings
num_patients = 3000
start_date = datetime(2026, 1, 30, 8, 0, 0)

# Categories
segments = ['IP (Inpatient)', 'OP (Outpatient)', 'HC (Health Check)']
departments = ['X-Ray', 'CT Scan', 'Pathology', 'Consultation', 'ECG']
urgency_levels = ['STAT', 'Normal', 'Fasting']
statuses = ['Completed', 'Completed', 'Completed', 'Completed', 'Waiting'] # mostly completed

data = []

print("Generating 3,000 rows of patient data...")

for i in range(num_patients):
    # 1. Identifier
    p_id = f"P-{1001 + i}"

    # 2. Segment & Urgency Logic
    # Weighted choice: OP is most common
    segment = random.choices(segments, weights=[15, 60, 25], k=1)[0]

    # Determine Urgency based on Segment (Logic for "Clinical Sequencing")
    if segment == 'IP (Inpatient)':
        urgency = random.choices(['STAT', 'Normal'], weights=[80, 20])[0]
    elif segment == 'HC (Health Check)':
        urgency = random.choices(['Fasting', 'Normal'], weights=[40, 60])[0]
    else: # OP
        urgency = 'Normal'

    department = random.choice(departments)

    # 3. Priority Score & Wait Time Logic (The Max-Heap Simulation)
    # High Priority = Low Wait Time
    if urgency == 'STAT':
        priority_score = random.randint(90, 100)
        wait_time = random.randint(0, 5) # Emergencies wait 0-5 mins
    elif segment == 'IP (Inpatient)':
        priority_score = random.randint(70, 89)
        wait_time = random.randint(5, 15)
    elif segment == 'OP (Outpatient)':
        priority_score = random.randint(40, 69)
        wait_time = random.randint(15, 60)
    else: # HC (Buffer)
        priority_score = random.randint(10, 39)
        wait_time = random.randint(30, 90) # Buffer patients wait longest

    # 4. Timestamps
    # Stagger arrival times over 3 days (approx 8AM to 8PM each day)
    day_offset = random.choice([0, 1, 2])
    time_offset_mins = random.randint(0, 720) # 12 hours window

    entry_time = start_date + timedelta(days=day_offset, minutes=time_offset_mins)

    # Service Start
    service_start = entry_time + timedelta(minutes=wait_time)

    # Service Duration & Exit
    service_duration = random.randint(5, 25)
    exit_time = service_start + timedelta(minutes=service_duration)

    # Status Check
    status = random.choice(statuses)
    if status == 'Waiting':
        # If waiting, they haven't started service yet
        service_start = None
        exit_time = None
        # Recalculate current wait time relative to "now" (simulated)
        wait_time = random.randint(5, 120)

    # Formatting Times for CSV (HH:MM AM/PM)
    def format_time(t):
        return t.strftime("%I:%M %p") if t else ""

    data.append([
        p_id,
        segment,
        urgency,
        format_time(entry_time),
        format_time(service_start),
        format_time(exit_time),
        wait_time,
        priority_score,
        department,
        status
    ])

# Create DataFrame
columns = [
    'Patient_ID',
    'Segment_Type',
    'Urgency_Level',
    'Entry_Time',
    'Service_Start',
    'Exit_Time',
    'Wait_Time_Mins',
    'Priority_Score',
    'Department',
    'Status'
]

df = pd.DataFrame(data, columns=columns)

# Save to CSV
file_name = 'Apollo_DQMS_3000_Rows.csv'
df.to_csv(file_name, index=False)

print(f"SUCCESS! Dataset with {num_patients} rows generated.")
print(f"File saved as: {file_name}")

# Logic to auto-download if running in Google Colab
try:
    from google.colab import files
    files.download(file_name)
except ImportError:
    print("Check your local folder for the CSV file.")


Generating 3,000 rows of patient data...
SUCCESS! Dataset with 3000 rows generated.
File saved as: Apollo_DQMS_3000_Rows.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>